# Prepare CVXLab inputs

Install MARIO and the optional `cvxlab` package in the same environment. Download `cvxlab_sut_settings.xlsx` from the accompanying guide and save it beside this notebook. The settings declare ten SUT input tables and a small feasibility problem for checking workbook loading.

In [ ]:
from pathlib import Path

import mario

settings_file = Path("cvxlab_sut_settings.xlsx")
source = mario.load_test("SUT")
db = source.to_single_region(
    source.regions[0], imports_by="commodity",
    import_representation="separate", inplace=False,
)

## Map coordinates and input tables

Left-hand names identify MARIO flat coordinates and matrices. Right-hand names identify CVXLab sets and exogenous data tables in the downloaded settings.

In [ ]:
mapping = {
    "sets": {
        "Region_from": "region_from",
        "Region_to": "region_to",
        "Activity_from": "activity_from",
        "Activity_to": "activity_to",
        "Commodity_from": "commodity_from",
        "Commodity_to": "commodity_to",
        "Consumption category_to": "consumption",
        "Factor of production_from": "factor",
        "Satellite account_from": "extension"
    },
    "matrices": {
        "S": "SUPPLY",
        "U": "USE",
        "Yc": "FINAL_DEMAND",
        "Va": "VALUE_ADDED",
        "VY": "VALUE_ADDED_FINAL_DEMAND",
        "IMP_a": "IMPORTS_USE",
        "IMP_c": "IMPORTS_SUPPLY",
        "IMP_Y": "IMPORTS_FINAL_DEMAND",
        "Ea": "EMISSIONS",
        "EY": "EMISSIONS_FINAL_DEMAND"
    }
}

In [ ]:
report = db.to_cvxlab(
    model_settings=settings_file,
    path="cvxlab_example",
    scenarios="baseline",
    mapping=mapping,
)
report

In [ ]:
assert not report["pending_sets"]
assert not report["pending_matrices"]
assert not report["unmatched_coordinates"]
assert not report["missing_values"]
assert report["written_cells"]["IMPORTS_USE"] == db.IMP_a.size
assert report["written_cells"]["IMPORTS_FINAL_DEMAND"] == db.IMP_Y.size
assert report["sets_file"].is_file()
assert all(path.is_file() for path in report["input_files"])

## Complete additional model inputs

For your own model, inspect `pending_sets`, `missing_attributes`, `pending_matrices`, `unmapped_tables`, `unmatched_coordinates` `missing_values` and `extra_input_rows`. Fill the additional information in the generated workbooks and repeat the call. Set updates append new members and preserve attributes. Mapped matrix cells are updated by coordinate. For several scenarios, define the scenario coordinate in the CVXLab settings and add `"Scenario": "scenario"` to the set mapping.

## Several matrices in one input table

For a model that defines shared tables, map several MARIO matrices to the same destination. For example, `"U": "use_0"` and `"Yc": "use_0"` write intermediate use and final demand into one table. Map `Activity_to` and `Consumption category_to` to the same CVXLab account set. Likewise, `IMP_a` and `IMP_Y` can populate `imports_0`, while `Va`, `VY`, `Ea` and `EY` can populate `extensions_0`. Map `Factor of production_from` and `Satellite account_from` to the shared extension set.

The accompanying guide includes the complete mapping for this table layout. The settings for that layout must declare the shared sets and tables. Overlapping source coordinates cause an explicit error before file updates.